# Assignment 1 - Gathering data by downloading a file

## Topic 19 - The Data Analytics Process

### Scenario

You have been asked to prepare a short evidence brief about how one indicator has changed over time in Ghana compared with two other countries. Nobody will hand you a spreadsheet. You have to find the data at a public URL, download it with Python, keep a record of where it came from, clean it, save it, and produce two charts that support your conclusion.

This is the ordinary first day of an analyst's project, and it is the whole point of this assignment. A polished chart built on data of unknown origin is worth nothing. A modest chart built on data whose source, retrieval date, and cleaning steps are all documented is professional work.

### What you are practising

| Skill | Where it appears |
|---|---|
| Downloading a file with `urllib.request` | Part 3 |
| Downloading the same file with `requests` | Part 3 |
| Writing your own reusable functions | Every part |
| Recording provenance with a checksum | Part 4 |
| Inspecting and cleaning data with pandas | Parts 5 and 6 |
| Separating raw data from cleaned data | Part 7 |
| Grouping and summarising with pandas | Part 8 |
| Line and bar charts with matplotlib | Part 9 |
| Submitting work with Git and GitHub | Part 12 |

## What you must submit

A GitHub repository named **`itfy-data-gathering-download`** containing:

1. this notebook, completed, **run from top to bottom so all outputs and charts are visible**;
2. `data/raw/` - the file exactly as you downloaded it, unchanged;
3. `data/clean/` - the cleaned dataset you saved;
4. `README.md` - your research question, source details, findings, and limitations;
5. a manifest file recording where the data came from.

Part 12 of this notebook explains exactly how to submit. If you have never used GitHub, work through [the GitHub submission guide](../topic19d_appendix_submitting_with_github.ipynb) first - it assumes no previous experience and is written for Windows.

### Rules

- **Every function skeleton must be completed.** Do not replace a function with loose code in a cell. Writing functions is being assessed directly.
- **Do not edit the raw downloaded file.** All cleaning happens in pandas and is saved to a separate file.
- **Charts are limited to line and bar** for the graded parts. Part 10 is optional space to explore other chart types if you wish.
- **Explain your results in words.** Every part has a markdown cell for your explanation. A correct number with no interpretation earns half the marks.
- **No API keys, passwords, or personal data** anywhere in the repository.

## How to work through this notebook

Each part follows the same three-step shape:

1. **Read** the explanation and the worked example.
2. **Complete** the function skeleton. Anywhere you see `# TODO`, replace it with your code. Anywhere you see `raise NotImplementedError(...)`, delete that line once your function works.
3. **Run** the check cell underneath, which tells you whether your function behaves as expected.

Work in order. Later parts use the variables and functions you create in earlier ones. If a cell fails, read the error message before changing anything - the message almost always names the problem.

---
## Part 1 - Set up your workspace

Run the cell below without changing it. It imports what you need and creates the folders your work will live in.

Two things to notice, because you will need both in your own projects:

- `pathlib.Path` builds file paths that work on Windows, macOS, and Linux. Writing `DATA_DIR / "raw"` is correct everywhere, whereas typing `"data\raw"` breaks on anything that is not Windows and can misbehave even there because `\r` means something special in Python strings.
- The `try`/`except ImportError` around `requests` means this notebook still runs if `requests` is not installed. You will install it in Part 3.

In [ ]:
# Run this cell as it is. Do not change it.
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import hashlib
import json

import pandas as pd
import matplotlib.pyplot as plt

# requests is optional at this point. Part 3 shows you how to install it.
try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False

# ---- Folders -----------------------------------------------------------------
# Everything is written relative to this notebook, so the repository you submit
# is self-contained and runs on somebody else's computer.
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"        # never edit anything in here
CLEAN_DIR = DATA_DIR / "clean"    # your cleaned output goes here
CHART_DIR = BASE_DIR / "charts"   # saved chart images, if you save any

for folder in (RAW_DIR, CLEAN_DIR, CHART_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# ---- Display settings --------------------------------------------------------
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110

# A polite identifier. Some servers reject requests that do not say who they are.
USER_AGENT = "ITFY-Ghana-student-assignment/1.0"

print("pandas version:     ", pd.__version__)
print("requests available: ", REQUESTS_AVAILABLE)
print("raw data folder:    ", RAW_DIR)
print("clean data folder:  ", CLEAN_DIR)

---
## Part 2 - Choose your dataset and write your question

Three datasets are offered below. **Choose one.** All three are public, free, published by a recognised organisation, and small enough to download in seconds.

### Option A - Gapminder five-year country indicators

| Item | Detail |
|---|---|
| Direct file URL | `https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv` |
| Publisher | Gapminder Foundation, distributed in the Plotly datasets collection |
| Size | About 82 KB, 1,704 rows, 6 columns |
| Columns | `country`, `year`, `pop`, `continent`, `lifeExp`, `gdpPercap` |
| What one row is | One country in one year |
| Coverage | 142 countries, 1952 to 2007, every five years |
| Documentation | https://www.gapminder.org/data/documentation/ |
| Cleaning you will face | Column names are abbreviated and inconsistent; population is stored as a float |

### Option B - Our World in Data life expectancy

| Item | Detail |
|---|---|
| Direct file URL | `https://ourworldindata.org/grapher/life-expectancy.csv` |
| Publisher | Our World in Data, based on the UN World Population Prospects and historical sources |
| Size | About 590 KB, 21,565 rows, 4 columns |
| Columns | `Entity`, `Code`, `Year`, `Life expectancy` |
| What one row is | One entity in one year |
| Coverage | Countries and regions, 1543 to 2023 |
| Documentation | https://ourworldindata.org/life-expectancy |
| Cleaning you will face | `Entity` mixes countries with aggregates such as `Africa` and `World`; those aggregate rows have a missing `Code`. Early years are sparse historical estimates. Column names contain spaces and capital letters. |

### Option C - World Bank GDP by country and year

| Item | Detail |
|---|---|
| Direct file URL | `https://raw.githubusercontent.com/datasets/gdp/main/data/gdp.csv` |
| Publisher | World Bank development indicators, redistributed by the Frictionless Data project |
| Size | About 550 KB, 13,979 rows, 4 columns |
| Columns | `Country Name`, `Country Code`, `Year`, `Value` |
| What one row is | One country in one year |
| Coverage | 1960 to 2023, GDP in current US dollars |
| Documentation | https://datahub.io/core/gdp |
| Cleaning you will face | The measure is called `Value`, which tells the reader nothing - you must rename it and record its unit. Values are very large, so a chart is unreadable until you convert to billions. `Country Name` includes aggregates such as `World` and `Euro area`. |

### Choosing well

Option A is the gentlest. Option B has the most interesting cleaning problem, because separating real countries from regional aggregates is a judgement you have to make and defend. Option C forces you to think about units, which is where a lot of real analysis goes wrong.

Whichever you choose, your question must **compare Ghana with at least two other countries over time**, because Part 9 asks for a line chart of change and a bar chart of a ranking.

Good question: *How did life expectancy in Ghana change between 1960 and 2023 compared with Kenya and Senegal?*

Poor question: *What is in this dataset?* That is not a question, it is a description.

In [ ]:
# The three options are defined for you. Read them, then set CHOSEN_OPTION below.
DATASET_OPTIONS = {
    "A": {
        "name": "Gapminder five-year country indicators",
        "url": "https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv",
        "raw_filename": "gapminder_five_year_raw.csv",
        "publisher": "Gapminder Foundation (via the Plotly datasets collection)",
        "documentation": "https://www.gapminder.org/data/documentation/",
        "row_meaning": "one country in one year",
        "country_column": "country",
        "year_column": "year",
        "value_column": "lifeExp",
        "value_unit": "years",
    },
    "B": {
        "name": "Our World in Data life expectancy",
        "url": "https://ourworldindata.org/grapher/life-expectancy.csv",
        "raw_filename": "owid_life_expectancy_raw.csv",
        "publisher": "Our World in Data",
        "documentation": "https://ourworldindata.org/life-expectancy",
        "row_meaning": "one entity in one year",
        "country_column": "Entity",
        "year_column": "Year",
        "value_column": "Life expectancy",
        "value_unit": "years",
    },
    "C": {
        "name": "World Bank GDP by country and year",
        "url": "https://raw.githubusercontent.com/datasets/gdp/main/data/gdp.csv",
        "raw_filename": "world_bank_gdp_raw.csv",
        "publisher": "World Bank (redistributed by Frictionless Data / datahub.io)",
        "documentation": "https://datahub.io/core/gdp",
        "row_meaning": "one country in one year",
        "country_column": "Country Name",
        "year_column": "Year",
        "value_column": "Value",
        "value_unit": "current US dollars",
    },
}

# TODO 1: Choose your dataset. Replace None with "A", "B", or "C".
CHOSEN_OPTION = None

# TODO 2: Write your research question as a full sentence ending in a question mark.
RESEARCH_QUESTION = ""

# TODO 3: Name Ghana plus at least two comparison countries.
#         Use the exact spelling that appears in your dataset - check it in Part 5.
FOCUS_COUNTRIES = ["Ghana"]

# TODO 4: State the range of years you will analyse.
START_YEAR = None
END_YEAR = None

# TODO 5: In one sentence, what do you expect to find, and why?
EXPECTATION = ""

# ---- Do not change below this line -------------------------------------------
if CHOSEN_OPTION not in DATASET_OPTIONS:
    raise ValueError('Set CHOSEN_OPTION to "A", "B", or "C" before continuing.')

DATASET = DATASET_OPTIONS[CHOSEN_OPTION]
SOURCE_URL = DATASET["url"]
RAW_PATH = RAW_DIR / DATASET["raw_filename"]

print("Dataset:  ", DATASET["name"])
print("URL:      ", SOURCE_URL)
print("Raw file: ", RAW_PATH)
print("Question: ", RESEARCH_QUESTION or "*** still empty ***")
print("Countries:", FOCUS_COUNTRIES)
print("Years:    ", START_YEAR, "to", END_YEAR)

---
## Part 3 - Download the file, two ways

A URL is just an address. When you type a data URL into a browser, the browser downloads bytes and shows them to you. Python can request exactly the same bytes, and this is better than clicking a download button, because code can be re-run, read by somebody else, and checked.

You will do this twice, with the two tools Python analysts actually use.

| | `urllib.request` | `requests` |
|---|---|---|
| Comes with Python | Yes, always available | No, must be installed |
| Typical lines of code | More | Fewer |
| Reading the status code | `response.status` | `response.status_code` |
| Getting the body as bytes | `response.read()` | `response.content` |
| Getting the body as text | `response.read().decode("utf-8")` | `response.text` |
| Parsing JSON | `json.load(response)` | `response.json()` |
| Errors | Raises `HTTPError` for 404, 500, and similar | Returns normally; you call `response.raise_for_status()` |

Neither is wrong. `urllib` is guaranteed to exist, which matters on a locked-down work computer. `requests` is shorter and clearer, which is why most teams install it.

### Installing requests

If the setup cell printed `requests available: False`, open a terminal and run:

```text
pip install requests
```

In VS Code you can also run it from a notebook cell by putting an exclamation mark in front: `!pip install requests`. After installing, restart the kernel and run the setup cell again.

### Worked example - the shape of both

Read this example carefully. It downloads a small file with each tool. You are not asked to change it, and you may run it: it writes to a scratch filename that is not part of your submission.

In [ ]:
# WORKED EXAMPLE - read this, run it, then write your own versions below.
EXAMPLE_URL = "https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv"
EXAMPLE_PATH = RAW_DIR / "_example_download.csv"

# ---- With urllib.request (standard library) ----------------------------------
# Request() lets us attach headers. urlopen() sends it and returns a response
# object that behaves like an open file, so we .read() it to get bytes.
example_request = Request(EXAMPLE_URL, headers={"User-Agent": USER_AGENT})
with urlopen(example_request, timeout=30) as response:
    print("urllib status code: ", response.status)
    print("urllib content type:", response.headers.get("Content-Type"))
    downloaded_bytes = response.read()

# "wb" means write bytes. We save exactly what the server sent, with no
# decoding or editing, because this file is our evidence of what we received.
EXAMPLE_PATH.write_bytes(downloaded_bytes)
print(f"urllib saved {len(downloaded_bytes):,} bytes to {EXAMPLE_PATH.name}")

# ---- With requests (third-party) --------------------------------------------
if REQUESTS_AVAILABLE:
    response = requests.get(EXAMPLE_URL, headers={"User-Agent": USER_AGENT}, timeout=30)
    response.raise_for_status()   # turns a 404 or 500 into an exception
    print("\nrequests status code: ", response.status_code)
    print("requests content type:", response.headers.get("Content-Type"))
    print(f"requests received {len(response.content):,} bytes")
else:
    print("\nrequests is not installed yet - see the instructions above.")

# ---- The first two lines of what we received --------------------------------
print("\nFirst two lines of the file:")
for line in downloaded_bytes.decode("utf-8").splitlines()[:2]:
    print("   ", line)

### Now write your own download functions

Complete the two functions below. They must behave identically apart from the tool they use.

Both functions should:

1. skip the download if the file already exists and `force` is `False`, and print that it is using the cached copy - this is politeness to the server and speed for you;
2. send the `User-Agent` header;
3. use a timeout, so a dead server cannot hang your notebook forever;
4. check that the download succeeded before writing anything;
5. write the bytes to `destination` **without modifying them**;
6. return a dictionary describing what happened.

Notice that the functions **return** their result rather than printing it. A function that only prints cannot be used by the next step of your pipeline. This is the single most common mistake beginners make when writing functions, and Part 4 depends on you getting it right.

In [ ]:
def download_with_urllib(url, destination, force=False, timeout=30):
    """Download a file using the standard library and return provenance details.

    Parameters
    ----------
    url : str
        The address of the file to download.
    destination : Path or str
        Where to save the file on this computer.
    force : bool
        If False and the file already exists, do not download it again.
    timeout : int
        Seconds to wait before giving up.

    Returns
    -------
    dict
        With the keys: "tool", "url", "path", "bytes", "status", "retrieved_utc",
        and "from_cache".
    """
    destination = Path(destination)

    # TODO 6: If the file already exists and force is False, print a message
    #         saying you are using the cached copy, and return a dictionary with
    #         "from_cache" set to True. Use destination.stat().st_size for the
    #         byte count, and set "status" to None because no request was made.

    # TODO 7: Build a Request with the User-Agent header, open it with urlopen
    #         using the timeout, read the bytes, and record response.status.
    #         Wrap this in try/except to catch HTTPError and URLError, and
    #         re-raise a clear RuntimeError explaining which URL failed.

    # TODO 8: Write the bytes to destination with .write_bytes().

    # TODO 9: Return the dictionary described in the docstring. Use
    #         datetime.now(timezone.utc).isoformat() for "retrieved_utc".

    raise NotImplementedError("Complete TODOs 6 to 9, then delete this line.")


def download_with_requests(url, destination, force=False, timeout=30):
    """Download a file using the requests library and return provenance details.

    Returns the same dictionary shape as download_with_urllib, so the two are
    interchangeable.
    """
    if not REQUESTS_AVAILABLE:
        raise RuntimeError("requests is not installed. Run: pip install requests")

    destination = Path(destination)

    # TODO 10: Same caching behaviour as TODO 6.

    # TODO 11: Call requests.get() with the header and timeout, then call
    #          response.raise_for_status() so a bad status becomes an error.

    # TODO 12: Save response.content to destination, and return the same
    #          dictionary shape, with "tool" set to "requests" and "status"
    #          set to response.status_code.

    raise NotImplementedError("Complete TODOs 10 to 12, then delete this line.")

In [ ]:
# TODO 13: Download YOUR chosen dataset using download_with_urllib, saving it
#          to RAW_PATH. Store the returned dictionary in a variable called
#          urllib_provenance, then print it.

# TODO 14: Download the SAME file with download_with_requests, but save it to a
#          different filename so you can compare the two. A good name is
#          RAW_DIR / ("requests_" + DATASET["raw_filename"]).
#          Store the result in requests_provenance.

# TODO 15: Both files should be byte-for-byte identical, because the tool used
#          to download a file cannot change the file. Prove it, by comparing
#          the "bytes" value from each dictionary and printing True or False.

# YOUR CODE HERE

### Your answer - compare the two tools

Complete this table from your own experience of writing the two functions. Replace the bracketed prompts with your own words.

| Question | Your answer |
|---|---|
| Which function took more lines of code, and why? | [your answer] |
| How did each one tell you the request had succeeded? | [your answer] |
| Which would you use on a computer where you cannot install packages, and why? | [your answer] |
| Which produced the clearer error message when something went wrong? | [your answer] |
| Did the two downloads produce identical files? What does that tell you? | [your answer] |

**Then try this deliberately.** Change one character in the URL so that it is wrong, call one of your functions, and read the error. Write down below what the error said and which line of your function raised it. Handling a broken source is a normal part of gathering data, and an analyst who has never seen the error message will not recognise it under pressure.

[your answer]

---
## Part 4 - Record where the data came from

You now have a file. In six months, will you remember which URL produced it, or which day you downloaded it? The file itself does not record either fact.

**Provenance** is the record of where data came from. At minimum, keep the source URL, the retrieval time, the local filename, the publisher, and a **checksum**.

A checksum is a short fingerprint calculated from the contents of a file. The SHA-256 algorithm reads every byte and produces a 64-character string. Change a single digit anywhere in the file and the fingerprint changes completely. This is what lets you prove that the file you are analysing today is the same file you downloaded last month - useful when a publisher revises a dataset without changing its filename.

A checksum does not tell you the data is *correct*. It only tells you the data is *unchanged*.

In [ ]:
def file_fingerprint(path):
    """Return the SHA-256 checksum of a file as a 64-character string.

    Read the file in chunks rather than all at once, so that this function works
    on a file larger than the computer's memory.
    """
    # TODO 16: Create a hash object with hashlib.sha256().
    # TODO 17: Open the file in binary mode ("rb"), read it in chunks of
    #          65536 bytes in a loop, and call .update(chunk) on each chunk.
    # TODO 18: Return the result of .hexdigest().
    raise NotImplementedError("Complete TODOs 16 to 18, then delete this line.")


def describe_source(provenance, dataset, checksum):
    """Combine download details, dataset details, and a checksum into one record.

    Returns
    -------
    dict
        A flat dictionary suitable for a manifest, containing at least:
        "source_name", "source_url", "publisher", "documentation",
        "local_file", "retrieved_utc", "file_bytes", "sha256", "row_meaning".
    """
    # TODO 19: Build and return that dictionary, taking values from the
    #          provenance dictionary your download function returned and from
    #          the dataset dictionary defined in Part 2.
    raise NotImplementedError("Complete TODO 19, then delete this line.")

In [ ]:
# TODO 20: Calculate the checksum of your raw file and print it.
# TODO 21: Build a source record with describe_source() and display it. A neat
#          way to display a single dictionary as a table is:
#              pd.Series(source_record).to_frame("value")
# TODO 22: Run file_fingerprint() on the raw file a second time and confirm you
#          get the same string. Explain in the next cell why that had to happen.

# YOUR CODE HERE

### Your answer - provenance

1. Paste the first 12 characters of your checksum here: [your answer]
2. Why did calculating the checksum twice give the same answer? [your answer]
3. If the publisher quietly corrected an error in this dataset next month and you downloaded it again under the same filename, how would the checksum help you notice? [your answer]
4. Name one thing a checksum **cannot** tell you about your data. [your answer]

---
## Part 5 - Load and inspect the raw data

Before cleaning anything, find out what you actually have. Resist the urge to plot. An analyst who charts data before understanding its grain will eventually publish a chart that means something other than what they claimed.

**Grain** is what one row represents. Get this wrong and every total you calculate is wrong. If one row is one country in one year, then summing the population column across the whole file adds the same country up 60 times and produces a meaningless number.

Your inspection must answer six questions:

1. How many rows and columns are there?
2. What is each column called, and what type is it?
3. What does one row represent?
4. Which values are missing, and in which columns?
5. Are there duplicate rows?
6. What range does each numeric column cover, and are the extremes plausible?

In [ ]:
def inspect_dataframe(df, name="dataset"):
    """Print a structured audit of a DataFrame and return a summary dictionary.

    Print, in this order and with clear labels:
      - the shape, as rows and columns
      - each column with its dtype and its count of missing values
      - the number of fully duplicated rows
      - describe() for the numeric columns
      - the first five rows

    Returns
    -------
    dict
        With the keys "rows", "columns", "column_names", "total_missing",
        and "duplicate_rows".
    """
    # TODO 23: Print the audit described above. Useful pieces:
    #            df.shape, df.dtypes, df.isna().sum(), df.duplicated().sum(),
    #            df.describe(), df.head()
    #          Give each section a heading so the output is readable.
    # TODO 24: Build and return the summary dictionary.
    raise NotImplementedError("Complete TODOs 23 and 24, then delete this line.")

In [ ]:
# TODO 25: Load your raw file into a DataFrame called raw_df with pd.read_csv().
#          Read from RAW_PATH, not from the internet - you already downloaded it.
# TODO 26: Run inspect_dataframe(raw_df, DATASET["name"]) and read the output.
# TODO 27: Find out which country names your dataset actually uses, so that
#          FOCUS_COUNTRIES in Part 2 matches them exactly. Print the unique
#          values of the country column that contain the text "Ghana". Then
#          check that every name in FOCUS_COUNTRIES is present, and print any
#          that are not. Fix FOCUS_COUNTRIES in Part 2 if a name is wrong.

# YOUR CODE HERE

### Your answer - what have you got?

1. What does exactly one row of your dataset represent? [your answer]
2. How many rows and columns does the raw file have? [your answer]
3. Which columns have missing values, and how many? [your answer]
4. Look at the largest and smallest values of your measure. Are they plausible for the real world, or does one of them suggest a data problem, an aggregate row, or a different unit than you assumed? [your answer]
5. Name one column whose meaning you could **not** work out from its name alone. What did you have to read to find out? [your answer]

---
## Part 6 - Clean the data

Cleaning is interpretation, and interpretation must be recorded. Every decision you make here changes what your charts mean, so each one belongs in your function and in your written notes.

Your `clean_data()` function must, at minimum:

1. **rename the columns** to clear, lowercase, underscore-separated names - `life_expectancy_years`, not `Life expectancy`, and certainly not `Value`;
2. **fix the data types** - a year should be an integer, a measure should be a float;
3. **restrict the rows to what your question needs** - your focus countries and your year range;
4. **remove rows that are not what you claim to be analysing** - if your dataset mixes countries with aggregates such as `World`, `Africa`, or `Euro area`, they must go, or your "country" ranking will be nonsense;
5. **handle missing values explicitly** - either drop them or keep them, but decide deliberately and say what you did and why;
6. **sort the result** by country and then year, so it is readable;
7. **add one derived column** that helps answer your question - for example the change since the first year, a value converted to billions, or a rank within each year.

A note on missing values. A missing measurement is not zero. Zero is a measurement that happens to be zero. If you replace missing life expectancy with 0, you have invented a country where newborns die immediately, and your average is corrupted. Dropping the row is usually honest; filling it with an invented number rarely is.

Notice that `clean_data()` starts with `df = df.copy()`. Without it, your function would modify the raw DataFrame that was passed in, and re-running the cell would give different results each time. This kind of quiet damage is very hard to debug, so copy first.

In [ ]:
# Guidance on renaming, per dataset option. Adjust to suit your own choice.
COLUMN_RENAMES = {
    "A": {"country": "country", "year": "year", "pop": "population",
          "continent": "continent", "lifeExp": "life_expectancy_years",
          "gdpPercap": "gdp_per_capita_usd"},
    "B": {"Entity": "country", "Code": "country_code", "Year": "year",
          "Life expectancy": "life_expectancy_years"},
    "C": {"Country Name": "country", "Country Code": "country_code",
          "Year": "year", "Value": "gdp_current_usd"},
}


def clean_data(df, countries, start_year, end_year, renames):
    """Return a cleaned copy of the raw data, ready for analysis.

    Parameters
    ----------
    df : pd.DataFrame
        The raw data. This function must not modify it.
    countries : list of str
        Keep only these countries.
    start_year, end_year : int
        Keep only years within this range, inclusive.
    renames : dict
        Mapping of old column name to new column name.

    Returns
    -------
    pd.DataFrame
        Cleaned, filtered, sorted, with a fresh index.
    """
    df = df.copy()   # never modify the caller's DataFrame

    # TODO 28: Rename the columns using the renames mapping.

    # TODO 29: Convert the year column to a plain integer and the measure
    #          column to float.

    # TODO 30: Keep only rows where the country is in `countries` and the year
    #          is between start_year and end_year inclusive.

    # TODO 31: Remove any rows that are aggregates rather than countries, if
    #          your dataset contains them. For option B, aggregate rows have a
    #          missing country_code, so df["country_code"].notna() is your test.
    #          If your dataset has no aggregates, write a comment saying so.

    # TODO 32: Handle missing values in your measure column deliberately.
    #          Write a comment stating what you did and why.

    # TODO 33: Drop exact duplicate rows, if any exist.

    # TODO 34: Add one derived column that helps answer your question.
    #          Example: value in the current year minus that country's earliest
    #          value, which you can get with a groupby and .transform("first").

    # TODO 35: Sort by country then year, reset the index with drop=True,
    #          and return the DataFrame.

    raise NotImplementedError("Complete TODOs 28 to 35, then delete this line.")

In [ ]:
# TODO 36: Call clean_data() and store the result in clean_df.
# TODO 37: Run inspect_dataframe(clean_df, "cleaned data") and compare it with
#          the raw audit from Part 5.
# TODO 38: Prove your cleaning worked, by checking all of these and printing a
#          clear True or False for each:
#            - the row count went down
#            - the column names are all lowercase with no spaces
#            - the year column is an integer type
#            - every country in clean_df is one of your FOCUS_COUNTRIES
#            - the year range is within START_YEAR to END_YEAR
#            - there are no missing values left in your measure column

# YOUR CODE HERE

### Your answer - cleaning decisions

Fill in this table. One row per decision you made, with the reason. This is the part of the assignment that separates an analyst from somebody who runs code.

| Decision | What I did | Why |
|---|---|---|
| Column names | [your answer] | [your answer] |
| Data types | [your answer] | [your answer] |
| Rows removed | [your answer] | [your answer] |
| Missing values | [your answer] | [your answer] |
| Derived column | [your answer] | [your answer] |

Then answer:

- How many rows did you start with, and how many remain? [your answer]
- Which single cleaning decision would change your final chart the most if you had chosen differently? [your answer]

---
## Part 7 - Save the cleaned data

You now hold two genuinely different things:

| | Raw | Clean |
|---|---|---|
| What it is | Evidence of what the source gave you | Your interpretation of that evidence |
| Where it lives | `data/raw/` | `data/clean/` |
| May you edit it? | Never | Yes, by re-running your cleaning code |
| If you lose it | Download it again from the URL | Rebuild it from the raw file |

Keeping them apart is what makes your work reproducible. Anybody with your raw file and your notebook can regenerate your clean file and check that they get what you got. If you overwrite the raw file with cleaned data, that check becomes impossible and your analysis has to be taken on trust.

Saving with `index=False` matters. Without it, pandas writes the row numbers as an extra unnamed column, and when somebody loads your file they get a mysterious `Unnamed: 0` column. You have probably already met this and wondered where it came from.

In [ ]:
def save_clean_data(df, destination):
    """Save a cleaned DataFrame to CSV and verify it can be read back.

    Returns
    -------
    dict
        With the keys "path", "rows", "columns", "bytes", and "round_trip_ok".
    """
    destination = Path(destination)

    # TODO 39: Save df to destination as CSV with index=False.

    # TODO 40: Read the file straight back into a new DataFrame. Compare its
    #          shape with df.shape and store the result as round_trip_ok.
    #          Reading back what you just wrote is a habit worth keeping: it
    #          catches an empty or half-written file immediately.

    # TODO 41: Return the dictionary described above. Get the file size with
    #          destination.stat().st_size.

    raise NotImplementedError("Complete TODOs 39 to 41, then delete this line.")

In [ ]:
# TODO 42: Build a clean filename that describes the contents, for example
#          CLEAN_DIR / "life_expectancy_ghana_comparison_clean.csv"
#          Avoid "clean_data.csv" and "final_final.csv".
# TODO 43: Call save_clean_data() and print the result.
# TODO 44: Confirm the round trip succeeded, and print the first three rows of
#          the file you read back.

# YOUR CODE HERE

---
## Part 8 - Analyse the cleaned data

Now, and only now, you may calculate something. Write three small functions, each doing one job. Small single-purpose functions are easier to test, easier to reuse, and much easier to debug than one long function that does everything.

In [ ]:
def summarise_by_country(df, country_column, value_column):
    """Return one row per country summarising the measure.

    Include, as columns: the number of observations, the first value, the last
    value, the change from first to last, the mean, the minimum, and the maximum.
    Sort by the last value, largest first.
    """
    # TODO 45: Group by country and aggregate. df.groupby(...).agg() accepts a
    #          dictionary or named aggregations such as
    #              first_value=(value_column, "first")
    #          Remember that "first" and "last" only mean what you want if the
    #          data is sorted by year, which your cleaning function guaranteed.
    # TODO 46: Add the change column, sort, and return the result.
    raise NotImplementedError("Complete TODOs 45 and 46, then delete this line.")


def top_n_in_year(df, year, value_column, n=5, year_column="year",
                  country_column="country"):
    """Return the n countries with the highest value in a single year.

    Keep only the country column and the value column, sorted highest first.
    """
    # TODO 47: Filter to the year, sort descending by value, take the head, and
    #          return it with a reset index.
    raise NotImplementedError("Complete TODO 47, then delete this line.")


def percent_change(df, country_column, year_column, value_column):
    """Return the percentage change in the measure for each country.

    The formula is (last - first) / first * 100.
    Return a DataFrame with the country, the first value, the last value, and
    the percentage change, sorted by the percentage change, largest first.
    """
    # TODO 48: Implement it. Guard against dividing by zero: if a first value
    #          is 0, the percentage change is undefined, so it should be NaN
    #          rather than infinity or a crash.
    raise NotImplementedError("Complete TODO 48, then delete this line.")

In [ ]:
# TODO 49: Run summarise_by_country() on clean_df and display the table.
# TODO 50: Run top_n_in_year() for the LAST year in your data. Do not type the
#          year as a number - calculate it with clean_df["year"].max(), so this
#          cell still works when the publisher adds another year.
# TODO 51: Run percent_change() and display the table.
# TODO 52: Answer this in code, printing the answer as a sentence: which of your
#          focus countries improved the most over your chosen period, and by
#          how much?

# YOUR CODE HERE

---
## Part 9 - Two charts

You need exactly two graded charts:

1. a **line chart** showing how your measure changed over time, with one line per country;
2. a **bar chart** ranking your countries by the measure in a single year.

Line for change over time, bar for comparison between categories. That is the whole rule, and choosing correctly between the two is most of chart selection in practice. A bar chart of 60 years of data is unreadable; a line chart comparing three countries in one year draws lines between things that have no order.

### Every chart you submit must have

- [ ] a title that states the finding, not the mechanics - `Life expectancy in Ghana, Kenya, and Senegal, 1960-2023`, not `Chart 1`
- [ ] an x-axis label including the unit
- [ ] a y-axis label including the unit
- [ ] a legend, whenever more than one series is drawn
- [ ] readable tick labels, rotated if they overlap
- [ ] a source note, so the reader knows where the data came from

That last point is a professional habit worth forming now. A chart that travels without its source is a rumour.

### Worked example

This example uses invented data, so it runs before your functions are finished and cannot be copied blindly into your answer. Read it for the pattern, then write your own.

In [ ]:
# WORKED EXAMPLE - invented numbers, shown for the pattern only.
example_df = pd.DataFrame({
    "country": ["Alpha"] * 4 + ["Beta"] * 4,
    "year": [2000, 2005, 2010, 2015] * 2,
    "score": [50.0, 54.5, 59.1, 63.2, 44.0, 49.8, 52.3, 58.9],
})

fig, ax = plt.subplots()

# One line per country. Looping over the groups keeps colours and the legend
# consistent, and works for any number of countries.
for country, group in example_df.groupby("country"):
    group = group.sort_values("year")          # a line chart must be in order
    ax.plot(group["year"], group["score"],
            marker="o", linewidth=2, label=country)

ax.set_title("Example score rose steadily in both countries, 2000-2015")
ax.set_xlabel("Year")
ax.set_ylabel("Score (points)")
ax.legend(title="Country")
ax.grid(True, alpha=0.3)                        # faint grid, easier to read
fig.text(0.99, -0.02, "Source: invented example data", ha="right", fontsize=8,
         color="grey")
fig.tight_layout()                              # stops labels being cut off
plt.show()

In [ ]:
def plot_line_trend(df, country_column, year_column, value_column,
                    value_label, title, source_note, save_path=None):
    """Draw a line chart of the measure over time, one line per country.

    Returns
    -------
    matplotlib.axes.Axes
        So the caller can adjust the chart further if needed.
    """
    fig, ax = plt.subplots()

    # TODO 53: Draw one line per country, sorting each group by year first.

    # TODO 54: Set the title, the axis labels including units, and the legend.

    # TODO 55: Add the source note, add a faint grid, and call fig.tight_layout().

    # TODO 56: If save_path was given, save the figure with
    #          fig.savefig(save_path, dpi=150, bbox_inches="tight") so your
    #          chart can also be viewed outside the notebook.

    # TODO 57: Return ax.

    raise NotImplementedError("Complete TODOs 53 to 57, then delete this line.")


def plot_bar_ranking(df, country_column, value_column, value_label, title,
                     source_note, save_path=None):
    """Draw a bar chart ranking countries by the measure.

    Expects one row per country, already sorted.
    """
    fig, ax = plt.subplots()

    # TODO 58: Draw the bars with ax.bar(). Label every bar with its value
    #          using ax.bar_label(bars, fmt="%.1f"), because a reader should not
    #          have to estimate a number from the height of a rectangle.

    # TODO 59: Set the title and the axis labels, rotate the tick labels if the
    #          country names overlap, add the source note, and tidy the layout.

    # TODO 60: Save if save_path was given, then return ax.

    raise NotImplementedError("Complete TODOs 58 to 60, then delete this line.")

In [ ]:
# TODO 61: Draw your line chart from clean_df. Save it into CHART_DIR.
#          Write a title that states what the chart shows, and a source note
#          naming the publisher and the retrieval date from your provenance.

# TODO 62: Draw your bar chart for the last year in your data, using the table
#          returned by top_n_in_year(). Save it into CHART_DIR too.

# YOUR CODE HERE

### Your answer - reading your own charts

1. State in one sentence what the line chart shows. Include at least two numbers. [your answer]
2. State in one sentence what the bar chart shows. Include at least two numbers. [your answer]
3. Why is a line chart the right choice for the first question and a bar chart for the second? [your answer]
4. Look hard at your line chart. Is there a point where a line changes direction sharply, or a gap where data is missing? What might explain it, and what would you need in order to check? [your answer]

---
## Part 10 - Optional: explore further

This part is not graded and is here because you should get into the habit of looking beyond what you were asked for.

Try one or more of the following, and write a sentence about what it showed you that the required charts did not:

- a **scatter plot** with `ax.scatter()`, if your dataset has two numeric measures, to look at whether they move together;
- a **horizontal bar chart** with `ax.barh()`, which is easier to read when country names are long;
- a **histogram** with `ax.hist()`, to see the distribution of your measure across all countries in one year rather than only your three;
- **small multiples** with `plt.subplots(1, 3)`, giving each country its own panel on a shared scale;
- an **annotation** with `ax.annotate()`, pointing at the single most interesting moment in your line chart.

Keep the same labelling standards. An exploratory chart with no axis labels is a note to yourself, and notes to yourself have a way of ending up in reports.

In [ ]:
# OPTIONAL - not graded. Explore here.

# YOUR CODE HERE

---
## Part 11 - Write your manifest

A **manifest** is a small machine-readable file that records everything about the data in your project. It is what turns a folder of files into a documented dataset. Include the raw source, the cleaned output, and the steps between them.

Save it as JSON, because JSON is readable by both people and programs, and commit it alongside your data.

In [ ]:
def build_manifest(source_record, clean_record, cleaning_notes, question,
                   countries, start_year, end_year):
    """Assemble a manifest describing this whole gathering exercise.

    Returns
    -------
    dict
        Structured with these top-level keys:
          "created_utc"      when the manifest was written
          "research_question"
          "focus_countries"
          "period"           a dict with "start" and "end"
          "raw_source"       the record from describe_source()
          "clean_output"     the record from save_clean_data()
          "cleaning_steps"   a list of strings, one per decision you made
          "tools"            a dict of library names to versions
    """
    # TODO 63: Build and return that dictionary. For "tools", include at least
    #          pandas, using pd.__version__.
    raise NotImplementedError("Complete TODO 63, then delete this line.")

In [ ]:
# TODO 64: Write your cleaning steps as a list of clear sentences. One entry per
#          decision from your Part 6 table, for example:
#            "Renamed 'Life expectancy' to 'life_expectancy_years'."
#            "Removed 1,348 rows where country_code was missing, because these
#             are regional aggregates rather than countries."
cleaning_notes = [
    # YOUR ENTRIES HERE
]

# TODO 65: Build the manifest, save it to DATA_DIR / "manifest.json" with
#          json.dumps(manifest, indent=2), and print the file contents so the
#          marker can see it in your notebook output.

# YOUR CODE HERE

---
## Part 12 - Your findings, and how to submit

### Write your findings

Answer in full sentences, in the cell below. Every claim needs a number attached to it.

1. **Answer your research question directly.** One paragraph, three to five sentences, containing at least three numbers from your analysis.
2. **The most surprising thing you found**, and why it surprised you.
3. **Three limitations of this data.** For each one, say what claim it stops you from making. "The data might be inaccurate" is not a limitation; "life expectancy before 1950 is a modelled estimate rather than a measurement, so I cannot treat the 1543 values as observations" is.
4. **What you would gather next**, and from which source, to strengthen this brief.
5. **One thing that went wrong** while you were doing this, and how you worked it out.

### Then submit

Full step-by-step instructions, written for Windows with no assumed knowledge, are in [the GitHub submission guide](../topic19d_appendix_submitting_with_github.ipynb). In summary:

1. Create a **public** GitHub repository named `itfy-data-gathering-download`.
2. Copy in this completed notebook, your `data/raw/` and `data/clean/` folders, your `charts/` folder, and your `manifest.json`.
3. Write a `README.md` using the template in the guide.
4. Commit with a clear message, then **push**.
5. Open your repository in a private browsing window to prove it is public.
6. Submit the repository link.

### Before you submit, check every line

- [ ] Every `# TODO` is complete and no `NotImplementedError` remains anywhere
- [ ] The notebook was run from top to bottom in order, and every output is visible
- [ ] Both charts appear, with titles, axis labels with units, legends, and source notes
- [ ] `data/raw/` holds the unmodified download, `data/clean/` holds the cleaned file
- [ ] `manifest.json` is present and complete
- [ ] Every "Your answer" cell is filled in - none still says `[your answer]`
- [ ] `README.md` is written
- [ ] The repository is public, and the link opens in a private window

### Your findings

**1. Answer to my research question:**

*(Write your answer here.)*

**2. The most surprising thing I found:**

*(Write your answer here.)*

**3. Three limitations of this data:**

1. *(Write your answer here.)*
2. *(Write your answer here.)*
3. *(Write your answer here.)*

**4. What I would gather next, and from where:**

*(Write your answer here.)*

**5. Something that went wrong, and how I worked it out:**

*(Write your answer here.)*

---
## Marking rubric

| Area | What earns the marks | Marks |
|---|---:|---:|
| Question and dataset choice | A specific, answerable question; the right dataset for it; sensible countries and period | 8 |
| Download functions | Both functions complete, with caching, headers, timeout, status checking, and a returned dictionary rather than a print | 16 |
| Comparing urllib and requests | The comparison table completed from experience, and a real error message investigated | 6 |
| Provenance and checksum | `file_fingerprint()` and `describe_source()` correct; provenance questions answered | 10 |
| Inspection | A thorough audit; the grain of the data correctly identified and stated | 10 |
| Cleaning | All required steps; aggregate rows handled; missing values handled deliberately; every decision justified in the table | 16 |
| Saving and raw/clean separation | Cleaned file saved with a descriptive name and verified by reading it back; raw file untouched | 6 |
| Analysis functions | All three functions correct and used; the improvement question answered in code | 10 |
| Charts | Line and bar charts correct, fully labelled with units and source notes, and interpreted in words | 12 |
| Manifest | Complete, valid JSON, cleaning steps recorded as readable sentences | 4 |
| Findings and limitations | Numbers in every claim; limitations that name the claim they block | 8 |
| Repository, README, and commits | Public repository, correct structure, complete README, commit messages that describe changes | 8 |
| **Total** | | **110** |

Marks are lost for: code that runs but is never explained, charts without axis units, a raw file that has been edited, a manifest that says `TODO`, `NotImplementedError` left in the notebook, and a repository that is not public.

## References

### Your data sources
- [Gapminder data documentation](https://www.gapminder.org/data/documentation/)
- [Our World in Data - life expectancy](https://ourworldindata.org/life-expectancy)
- [World Bank GDP dataset on DataHub](https://datahub.io/core/gdp)

### Downloading files
- [Python `urllib.request` documentation](https://docs.python.org/3/library/urllib.request.html)
- [`requests` quickstart guide](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [Python `hashlib` documentation](https://docs.python.org/3/library/hashlib.html)
- [Python `pathlib` documentation](https://docs.python.org/3/library/pathlib.html)

### pandas and matplotlib
- [`pandas.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [pandas group by user guide](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [matplotlib `plot` - line charts](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.plot.html)
- [matplotlib `bar` - bar charts](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.bar.html)
- [matplotlib quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html)

### Course material
- [Topic 19 - Data Gathering lesson](../topic19_data_gathering.ipynb)
- [Appendix - common data sources](../topic19b_appendix_common_data_sources.ipynb)
- [Appendix - reading dataset documentation](../topic19c_appendix_reading_dataset_documentation.ipynb)
- [Appendix - submitting your work with GitHub](../topic19d_appendix_submitting_with_github.ipynb)
- [Assignment 2 - gathering data from an API](topic19_assignment_api_gathering.ipynb)